In [ ]:
!pip install langchain langgraph langchain-google-genai faiss-cpu langchain_community

In [ ]:
from google.colab import userdata
from dotenv import load_dotenv
load_dotenv()

import os

# KEY CHANGE: Fetch Google API Key instead of OpenAI API Key
# In Colab: go to Secrets (🔑 icon) and add GOOGLE_API_KEY
# Get your key at: https://aistudio.google.com/app/apikey
google_api_key = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = google_api_key

print("Google API Key loaded successfully.")

In [ ]:
# KEY CHANGE: Import from langchain_google_genai instead of langchain_openai
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# KEY CHANGE: Use Gemini model instead of GPT-4o
# gemini-1.5-flash: fast and cost-efficient
# Alternative: gemini-1.5-pro for higher reasoning capability
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    convert_system_message_to_human=True  # Required: Gemini doesn't support system messages natively
)

# KEY CHANGE: Use Google's embedding model instead of OpenAI's
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001"
)

print("Gemini LLM and Embeddings initialized.")
print(f"LLM model   : gemini-1.5-flash")
print(f"Embeddings  : models/embedding-001")

In [ ]:
# NO CHANGE: FAISS vector store setup is identical
# The embeddings object now uses Google's model, but the interface is the same
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Automotive diagnostic knowledge base
docs = [
    Document(page_content="High engine temperature may indicate coolant leak or radiator failure."),
    Document(page_content="Low battery voltage often suggests battery degradation."),
    Document(page_content="Brake pressure loss is usually caused by hydraulic fluid leakage."),
    Document(page_content="Brake system faults are safety critical and require immobilization.")
]

# Build FAISS index using Google embeddings
vectorstore = FAISS.from_documents(docs, embeddings)

# Retrieve top-2 most relevant docs for any given anomaly
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Knowledge base created with {len(docs)} documents.")

In [ ]:
# NO CHANGE: State schema is model-agnostic
from typing import TypedDict, List, Dict

class VehicleState(TypedDict):
    vehicle_id: str              # Vehicle identifier
    telemetry: Dict[str, float]  # Sensor readings: engine_temp, battery_voltage, brake_pressure
    anomaly: str                 # Detected fault label (set by anomaly_node)
    retrieved_docs: List[str]    # Relevant knowledge from FAISS (set by retrieval_node)
    diagnosis: str               # LLM-generated root cause (set by diagnosis_node)
    decision: str                # LLM-recommended action (set by decision_node)

In [ ]:
# NO CHANGE: Pass-through node — just forwards initial state into the pipeline
def telemetry_node(state: VehicleState):
    return state

In [ ]:
# NO CHANGE: Rule-based anomaly detection — model independent
def anomaly_node(state: VehicleState):
    t = state["telemetry"]

    if t["engine_temp"] > 110:
        state["anomaly"] = "High engine temperature"
    elif t["battery_voltage"] < 11.5:
        state["anomaly"] = "Low battery voltage"
    elif t["brake_pressure"] < 20:
        state["anomaly"] = "Brake pressure loss"
    else:
        state["anomaly"] = "Normal"

    print(f"[anomaly_node] Detected: {state['anomaly']}")
    return state

In [ ]:
# NO CHANGE: Retrieval logic is model-agnostic
# The retriever now uses Google embeddings internally, but the API is identical
def retrieval_node(state: VehicleState):
    docs = retriever.invoke(state["anomaly"])
    state["retrieved_docs"] = [d.page_content for d in docs]
    print(f"[retrieval_node] Retrieved {len(state['retrieved_docs'])} docs.")
    return state

In [ ]:
# MINOR CHANGE: llm now uses Gemini instead of GPT-4o, but the call is identical
# Gemini's response format (.content) is compatible with LangChain's ChatModel interface
def diagnosis_node(state: VehicleState):
    context = "\n".join(state["retrieved_docs"])

    prompt = f"""
    You are an automotive diagnostics expert.

    Telemetry:
    {state["telemetry"]}

    Detected anomaly:
    {state["anomaly"]}

    Relevant knowledge:
    {context}

    Identify the most likely root cause.
    """

    # KEY CHANGE: llm is now ChatGoogleGenerativeAI (Gemini), but .invoke() interface is identical
    response = llm.invoke(prompt)
    state["diagnosis"] = response.content
    return state

In [ ]:
# MINOR CHANGE: llm now uses Gemini, but the call and response handling are identical
def decision_node(state: VehicleState):
    prompt = f"""
    You are a fleet safety AI.

    Diagnosis:
    {state["diagnosis"]}

    Decide the safest action.
    Constraints:
    - Human safety first
    - Vehicle protection second
    - Cost last

    Respond with ONE action.
    """

    # KEY CHANGE: llm is now ChatGoogleGenerativeAI (Gemini)
    response = llm.invoke(prompt)
    state["decision"] = response.content
    return state

In [ ]:
# NO CHANGE: Report node is model-agnostic
def report_node(state: VehicleState):
    print("\n===== VEHICLE INCIDENT REPORT =====")
    print("Vehicle ID:", state["vehicle_id"])
    print("Telemetry:", state["telemetry"])
    print("Anomaly:", state["anomaly"])
    print("Diagnosis:", state["diagnosis"])
    print("Decision:", state["decision"])
    print("=================================\n")
    return state

In [ ]:
# NO CHANGE: Graph assembly is model-agnostic
from langgraph.graph import StateGraph, END

graph = StateGraph(VehicleState)

# Register all nodes
graph.add_node("telemetry", telemetry_node)
graph.add_node("anomaly", anomaly_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("diagnose", diagnosis_node)
graph.add_node("decide", decision_node)
graph.add_node("report", report_node)

# Set entry point
graph.set_entry_point("telemetry")

# Define linear pipeline edges
graph.add_edge("telemetry", "anomaly")
graph.add_edge("anomaly", "retrieve")
graph.add_edge("retrieve", "diagnose")
graph.add_edge("diagnose", "decide")
graph.add_edge("decide", "report")
graph.add_edge("report", END)

# Compile the graph
app = graph.compile()

print("LangGraph pipeline compiled successfully.")

In [ ]:
# NO CHANGE: Test input and invocation are model-agnostic
initial_state = {
    "vehicle_id": "CAR-9001",
    "telemetry": {
        "engine_temp": 128,       # Triggers: High engine temperature (> 110)
        "battery_voltage": 12.1,  # Normal (>= 11.5)
        "brake_pressure": 30      # Normal (>= 20)
    },
    "anomaly": "",
    "retrieved_docs": [],
    "diagnosis": "",
    "decision": ""
}

# Invoke the full pipeline
result = app.invoke(initial_state)
result

In [ ]:
# BONUS: Test all three anomaly scenarios with Gemini

test_cases = [
    {
        "label": "Low Battery Voltage",
        "vehicle_id": "CAR-9002",
        "telemetry": {"engine_temp": 95, "battery_voltage": 10.8, "brake_pressure": 35}
    },
    {
        "label": "Brake Pressure Loss",
        "vehicle_id": "CAR-9003",
        "telemetry": {"engine_temp": 90, "battery_voltage": 12.5, "brake_pressure": 12}
    },
    {
        "label": "All Normal",
        "vehicle_id": "CAR-9004",
        "telemetry": {"engine_temp": 95, "battery_voltage": 12.6, "brake_pressure": 40}
    }
]

for tc in test_cases:
    print(f"\n{'='*50}")
    print(f"TEST SCENARIO: {tc['label']}")
    print(f"{'='*50}")
    state = {
        "vehicle_id": tc["vehicle_id"],
        "telemetry": tc["telemetry"],
        "anomaly": "",
        "retrieved_docs": [],
        "diagnosis": "",
        "decision": ""
    }
    app.invoke(state)

In [ ]:
# NO CHANGE: Graph visualization is model-agnostic
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))